In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

In [ ]:
def target_pdf(y):
    m_Z = 91.1876     
    Gamma_Z = 2.4952  
    
    # ---------------------------------------------------------
    # STEP 1: THE BOUNDING BOX (y -> Physical Variables)
    # ---------------------------------------------------------
    sig_y0 = torch.sigmoid(y[:, 0])
    sig_y1 = torch.sigmoid(y[:, 1])
    
    E = 50.0 + 100.0 * sig_y0
    cos_theta = -1.0 + 2.0 * sig_y1
    
    # ---------------------------------------------------------
    # STEP 2: TRACK THE SQUISH (HACK-PROOF JACOBIAN)
    # ---------------------------------------------------------
    # We calculate the exact log-derivative using F.logsigmoid to prevent numerical underflow.
    # No "+ 1e-10" safety nets allowed!
    log_jac_0 = math.log(100.0) + F.logsigmoid(y[:, 0]) + F.logsigmoid(-y[:, 0])
    log_jac_1 = math.log(2.0) + F.logsigmoid(y[:, 1]) + F.logsigmoid(-y[:, 1])
    
    log_bounding_jac = log_jac_0 + log_jac_1
    
    # ---------------------------------------------------------
    # STEP 3: THE QUANTUM MECHANICS 
    # ---------------------------------------------------------
    s = E**2
    bw_numerator = s
    bw_denominator = (s - m_Z**2)**2 + (m_Z * Gamma_Z)**2
    breit_wigner = bw_numerator / bw_denominator
    
    angular = 1.0 + cos_theta**2
    f_phys = breit_wigner * angular
    
    # ---------------------------------------------------------
    # STEP 4: FINAL COMBINATION
    # ---------------------------------------------------------
    # Note: f_phys inside our box [50, 150] NEVER mathematically reaches zero, 
    # so we don't even need a + 1e-15 here. It is naturally stable.
    log_f_y = torch.log(f_phys) + log_bounding_jac
    
    return log_f_y

In [ ]:
# ==========================================
# 2. THE MACHINE LEARNING ARCHITECTURE
# ==========================================
class AffineCouplingLayer(nn.Module):
    def __init__(self, dim, hidden_dim, mask):
        super().__init__()
        self.mask = mask
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.LeakyReLU(0.01),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.01),
            nn.Linear(hidden_dim, dim * 2) 
        )
        
        # --- THE FIX: Identity Initialization ---
        # Force the final layer to output exactly 0.0 at the start of training.
        # This guarantees s=0 and t=0, preventing the space from exploding.
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)

    def forward(self, z):
        z_A = z * self.mask
        s, t = self.mlp(z_A).chunk(2, dim=-1)
        
        # We can keep the clamp as an extra safety net
        s = torch.clamp(s, min=-5.0, max=5.0) 
        
        inv_mask = 1.0 - self.mask
        s = s * inv_mask
        t = t * inv_mask
        
        x = z_A + inv_mask * (z * torch.exp(s) + t)
        log_det = s.sum(dim=-1)
        return x, log_det

class NormalizingFlow(nn.Module):
    # SCALED UP: Now 16 layers deep, 128 neurons wide
    def __init__(self, dim=2, num_layers=8, hidden_dim=64):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            mask = torch.tensor([1.0, 0.0]) if i % 2 == 0 else torch.tensor([0.0, 1.0])
            self.layers.append(AffineCouplingLayer(dim, hidden_dim, mask))

    def forward(self, z):
        log_det_total = torch.zeros(z.shape[0], device=z.device)
        for layer in self.layers:
            z, log_det = layer(z)
            log_det_total += log_det
        return z, log_det_total

In [ ]:
def train_flow():
    dim = 2
    flow = NormalizingFlow(dim=dim, num_layers=8, hidden_dim=64)
    
    # Start with a slightly higher learning rate since the scheduler will shrink it
    optimizer = optim.Adam(flow.parameters(), lr=2e-3)
    
    epochs = 5000 
    batch_size = 8192 # Increased batch size for smoother gradients with a larger network
    
    # --- UPGRADE 1: Cosine Annealing Scheduler ---
    # Smoothly reduces the learning rate to 0 following a cosine curve
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    beta_start = 0.05       
    beta_end = 1.0          
    anneal_epochs = 2000
    
    # --- NEW: EARLY STOPPING TRACKERS ---
    best_loss = float('inf')
    patience_counter = 0
    patience_limit = 300  # How many epochs we tolerate no improvement
    
    print("Starting Training with Early Stopping Monitor...")
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        beta = beta_start + (beta_end - beta_start) * min(1.0, epoch / anneal_epochs)
            
        z = torch.randn(batch_size, dim)
        log_p_z = -0.5 * (z**2).sum(dim=-1) - (dim / 2.0) * math.log(2 * math.pi)
        
        x, log_det = flow(z)
        log_q_x = log_p_z - log_det
        log_f_x = target_pdf(x)
        
        loss = (log_q_x - beta * log_f_x).mean()
        loss.backward()
        
        # --- UPGRADE 2: Gradient Clipping ---
        # Prevents exploding gradients (NaNs) by capping the maximum gradient vector length at 1.0
        nn.utils.clip_grad_norm_(flow.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step() # Tell the scheduler to update the learning rate
        
        current_loss = loss.item()
        current_lr = scheduler.get_last_lr()[0]
        
        if epoch % 400 == 0:
            print(f"Epoch {epoch:4d} | Beta: {beta:.3f} | LR: {current_lr:.5f} | Loss: {current_loss:.4f}")
            
        # =========================================================
        # THE EARLY STOPPING MONITOR
        # =========================================================
        # Rule 1: Only start tracking AFTER the physics is fully sharpened
        if epoch > anneal_epochs:
            
            # Rule 2: Check if the loss genuinely improved
            if current_loss < best_loss - 1e-4:
                best_loss = current_loss
                patience_counter = 0  # Reset the clock!
            else:
                patience_counter += 1 # Tick the clock
                
            # Rule 3: Pull the plug if we hit the limit
            if patience_counter >= patience_limit:
                print(f"\n[EARLY STOPPING TRIGGERED]")
                print(f"Loss completely flatlined for {patience_limit} epochs.")
                print(f"Training successfully terminated early at Epoch {epoch} to save time.")
                break # This instantly shatters the for-loop

    return flow

In [ ]:
# ==========================================
# 4. UPGRADED PLOTTING (Physical Variables)
# ==========================================
def integrate_and_plot(flow, num_samples=50000000): # High samples for smooth physics
    print("\nStarting Physics Integration...")
    dim = 2
    flow.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, 2)
        log_p_z = -0.5 * (z**2).sum(dim=-1) - (dim / 2.0) * math.log(2 * math.pi)
        
        y, log_det = flow(z)
        log_q_y = log_p_z - log_det
        log_f_y = target_pdf(y)
        
        weights = torch.exp(log_f_y - log_q_y)
        
        integral = weights.mean().item()
        variance = weights.var().item()
        error = math.sqrt(variance / num_samples)
        
        print(f"Calculated Integral : {integral:.6f} +/- {error:.6f}")
        print(f"Variance (sigma^2)  : {variance:.6f}")
        
        # --- CONVERT NEURAL NET OUTPUTS TO PHYSICAL KINEMATICS FOR PLOTTING ---
        sig_y0 = torch.sigmoid(y[:, 0])
        sig_y1 = torch.sigmoid(y[:, 1])
        E = 50.0 + 100.0 * sig_y0
        cos_theta = -1.0 + 2.0 * sig_y1
        
        E_np = E.numpy()
        cos_np = cos_theta.numpy()
    
    
    m_Z = 91.1876
    # Generate the Dalitz-style Phase Space Plot
    plt.figure(figsize=(9, 6))
    # We zoom in on the E-axis [80, 105] to see the resonance clearly
    plt.hist2d(E_np, cos_np, bins=120, cmap='magma', range=[[80, 105], [-1, 1]])
    plt.colorbar(label='Generated Event Density')
    
    # Draw a line exactly where the Z mass is supposed to be
    plt.axvline(m_Z, color='white', linestyle='--', alpha=0.7, label=f'Z Mass ({m_Z} GeV)')
    
    plt.title("Learned Z Boson Resonance ($e^+ e^- \\to Z \\to \\mu^+ \\mu^-$)")
    plt.xlabel("Invariant Mass $E$ [GeV]")
    plt.ylabel("Scattering Angle $\\cos(\\theta)$")
    plt.legend()
    plt.show()

In [ ]:
trained_model = train_flow()

In [ ]:
integrate_and_plot(trained_model)